Memoria de Parejas - Agente Inteligente

Este cuadernillo está separado por secciones, igual que 01_introduccion y 01_01_tres_en_raya, para que sea fácil de explicar y ejecutar.
- Estado: qué cartas ya vio el agente y cuáles emparejó.
- Entorno: un tablero de 4 posiciones, numeradas de 0 a 3.
- Política: si conoce una pareja, la usa; si no, explora.
- Recompensa: 1 si encuentra pareja, 0 si no.
- Acción: elegir dos posiciones para abrir.

Crear un agente simple que aprenda a recordar cartas, encontrar parejas y luego jugar contra el usuario.

In [1]:
# Librerías necesarias para el juego y para guardar el agente
import numpy as np
import pickle
import random

Tablero del juego

Esta parte sirve para crear el juego. El tablero tiene 4 posiciones: 0, 1, 2 y 3.
Dos posiciones forman una pareja.

In [2]:
class Board:
    def __init__(self):
        # El juego se simplifica a 4 posiciones: 0, 1, 2 y 3
        self.n_positions = 4
        self.n_pairs = 2
        self.deck = list(range(self.n_pairs)) * 2
        random.shuffle(self.deck)
        self.matched = set()

    def reset(self):
        # Reinicia el tablero para una nueva partida
        self.deck = list(range(self.n_pairs)) * 2
        random.shuffle(self.deck)
        self.matched = set()

    def valid_moves(self):
        # Devuelve las posiciones que todavía no han sido emparejadas
        return [i for i in range(self.n_positions) if i not in self.matched]

    def reveal(self, pos):
        # Muestra qué carta hay en una posición
        return self.deck[pos]

    def match(self, pos1, pos2):
        # Marca una pareja si las dos cartas son iguales
        if self.deck[pos1] == self.deck[pos2]:
            self.matched.add(pos1)
            self.matched.add(pos2)
            return True
        return False

    def is_game_over(self):
        # Termina cuando ya se encontraron las 4 cartas
        return len(self.matched) == self.n_positions

In [3]:
class Agent:
    def __init__(self, alpha=0.5, prob_exp=0.5):
        # value_function guarda la memoria aprendida del agente
        self.value_function = {}
        self.alpha = alpha
        self.prob_exp = prob_exp
        self.positions = []

    def reset(self):
        # Limpia la memoria temporal de la partida actual
        self.positions = []

    def _board_state(self, board):
        # Estado simple: posiciones ya emparejadas
        return str(sorted(board.matched))

    def move(self, board, explore=True):
        valid = board.valid_moves()
        if explore and random.random() < self.prob_exp:
            # Explorar: elegir una posición oculta al azar
            return random.choice(valid)
        # Explotar: elegir una acción con mejor valor conocido
        max_val = -1e9
        best = valid[0]
        for pos in valid:
            board_copy = board.matched.copy()
            next_state = str(sorted(board_copy))
            val = self.value_function.get(next_state, 0)
            if val > max_val:
                max_val = val
                best = pos
        return best

    def update(self, board):
        # Guarda el estado actual para aprender después
        state = self._board_state(board)
        self.positions.append(state)

    def reward(self, r):
        # Actualiza la memoria del agente con la recompensa obtenida
        for p in reversed(self.positions):
            if p not in self.value_function:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (r - self.value_function[p])
            r = self.value_function[p]

Juego de entrenamiento

Esta parte sirve para que dos agentes jueguen entre sí y aprendan con self-play.

In [4]:
class Game:
    def __init__(self, player1, player2):
        self.player1 = player1
        self.player2 = player2
        self.board = Board()
        self.scores = [0, 0]

    def selfplay(self, rounds=1000):
        # Entrenamiento por self-play
        for episode in range(1, rounds + 1):
            self.board.reset()
            self.player1.reset()
            self.player2.reset()
            self.scores = [0, 0]
            current_player_idx = 0
            players = [self.player1, self.player2]

            while not self.board.is_game_over():
                player = players[current_player_idx]

                valid = self.board.valid_moves()
                if len(valid) < 2:
                    break

                pos1 = player.move(self.board)
                val1 = self.board.reveal(pos1)
                self.player1.update(self.board)
                self.player2.update(self.board)

                pos2 = player.move(self.board)
                val2 = self.board.reveal(pos2)
                self.player1.update(self.board)
                self.player2.update(self.board)

                if self.board.match(pos1, pos2):
                    self.scores[current_player_idx] += 1
                    self.player1.reward(1)
                    self.player2.reward(0)
                else:
                    self.player1.reward(0)
                    self.player2.reward(0)

                current_player_idx = 1 - current_player_idx

        return self.scores

 Entrenamiento del agente

Ejecuta esta celda para entrenar el agente y guardar su memoria en `agente_memoria.pickle`.

In [5]:
# Entrenamiento rápido del agente
agent1 = Agent(alpha=0.5, prob_exp=0.5)
agent2 = Agent()
game = Game(agent1, agent2)

scores = game.selfplay(3000)
print('Puntuaciones finales:', scores)

# Revisar qué estados aprendió
items = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)
print('Estados en tabla:', len(items))
print('Top valores:')
for s, v in items[:5]:
    print(f'  {v:.3f}: {s}')

# Mostrar una tabla pequeña para ver mejor el aprendizaje
import pandas as pd
funcion_de_valor = pd.DataFrame({
    'estado': [x[0] for x in items],
    'valor': [x[1] for x in items]
})
print('')
print('Tabla de los estados aprendidos:')
display(funcion_de_valor.head(10))

# Guardar la memoria aprendida del agente
with open('agente_memoria.pickle', 'wb') as f:
    pickle.dump(agent1.value_function, f, protocol=pickle.HIGHEST_PROTOCOL)
print('Agente guardado en agente_memoria.pickle')

Puntuaciones finales: [2, 1]
Estados en tabla: 11
Top valores:
  1.000: [1, 2]
  1.000: [2, 3]
  1.000: [0, 3]
  0.933: [3]
  0.859: [0, 1]

Tabla de los estados aprendidos:


,estado,valor
0,"[1, 2]",1.000000
1,"[2, 3]",1.000000
2,"[0, 3]",1.000000
3,[3],0.933386
4,"[0, 1]",0.859348
5,[2],0.805601
6,[1],0.784422
7,"[0, 2]",0.750000
8,"[1, 3]",0.750000
9,[],0.729591


Agente guardado en agente_memoria.pickle


In [6]:
class AgentInferencia:
    def __init__(self, value_function):
        self.value_function = value_function

    def move(self, board):
        valid = board.valid_moves()
        max_val = -1e9
        best = valid[0]
        for pos in valid:
            state = str(sorted(board.matched))
            val = self.value_function.get(state, 0)
            if val > max_val:
                max_val = val
                best = pos
        return best


def dibujar_tablero(board):
    print('Posiciones (0-3):', end=' ')
    for i in range(4):
        if i in board.matched:
            print(f'[{board.deck[i]}]', end=' ')
        else:
            print(f' {i} ', end=' ')
    print()


def movimiento_humano(board):
    while True:
        try:
            pos = int(input('Tu posición (0-3): '))
            if pos < 0 or pos > 3:
                print('Ingresa un número 0-3')
                continue
            if pos in board.matched:
                print('Esa posición ya está emparejada')
                continue
            return pos
        except ValueError:
            print('Ingresa un número válido')


def jugar(agente_empieza=False):
    try:
        with open('agente_memoria.pickle', 'rb') as f:
            value_func = pickle.load(f)
        print('Agente cargado desde agente_memoria.pickle')
    except FileNotFoundError:
        print('Primero ejecuta la celda de entrenamiento')
        return

    agente = AgentInferencia(value_func)
    board = Board()
    puntos_humano = 0
    puntos_agente = 0

    print('\nmemoria de parejas')
    print('Humano vs Agente')
    dibujar_tablero(board)

    turno_agente = agente_empieza
    while not board.is_game_over():
        if turno_agente:
            print('\nTurno agente:')
            pos1 = agente.move(board)
            val1 = board.reveal(pos1)
            print(f'  Abre posición {pos1} (valor {val1})')
            dibujar_tablero(board)

            pos2 = agente.move(board)
            val2 = board.reveal(pos2)
            print(f'  Abre posición {pos2} (valor {val2})')

            if board.match(pos1, pos2):
                print('  ¡Pareja encontrada!')
                puntos_agente += 1
            else:
                print('  No es pareja')
        else:
            print('\nTu turno:')
            pos1 = movimiento_humano(board)
            val1 = board.reveal(pos1)
            print(f'Cartas: posición {pos1} (valor {val1})', end='')
            dibujar_tablero(board)

            pos2 = movimiento_humano(board)
            val2 = board.reveal(pos2)
            print(f'Cartas: posición {pos2} (valor {val2})', end='')

            if board.match(pos1, pos2):
                print('¡Pareja encontrada!')
                puntos_humano += 1
            else:
                print('No es pareja')

        dibujar_tablero(board)
        turno_agente = not turno_agente

    print('\n game over!')
    print(f'Puntos - Humano: {puntos_humano}, Agente: {puntos_agente}')
    if puntos_humano > puntos_agente:
        print('¡Ganaste!')
    elif puntos_agente > puntos_humano:
        print('Gana el agente')
    else:
        print('Empate')

In [7]:
jugar(agente_empieza=False)

Agente cargado desde agente_memoria.pickle

memoria de parejas
Humano vs Agente
Posiciones (0-3):  0   1   2   3  

Tu turno:


Cartas: posición 0 (valor 0)Posiciones (0-3):  0   1   2   3  
Cartas: posición 2 (valor 0)¡Pareja encontrada!
Posiciones (0-3): [0]  1  [0]  3  

Turno agente:
  Abre posición 1 (valor 1)
Posiciones (0-3): [0]  1  [0]  3  
  Abre posición 1 (valor 1)
  ¡Pareja encontrada!
Posiciones (0-3): [0] [1] [0]  3  

Tu turno:
Esa posición ya está emparejada
Esa posición ya está emparejada
Esa posición ya está emparejada
Esa posición ya está emparejada
Cartas: posición 3 (valor 1)Posiciones (0-3): [0] [1] [0]  3  
Esa posición ya está emparejada
Esa posición ya está emparejada
Esa posición ya está emparejada
Cartas: posición 3 (valor 1)¡Pareja encontrada!
Posiciones (0-3): [0] [1] [0] [1] 

 game over!
Puntos - Humano: 2, Agente: 1
¡Ganaste!
